# 🎵 TuneWave Music Player — Milestone 2: Up Next & Playback History
### Core Data Structures: FIFO Queue (`collections.deque`) + Bounded LIFO Stack (Capacity = 10)

---

## 🎯 Milestone Goal & Problem Statement
In **TuneWave**, user listening sessions are non-linear. Users frequently queue songs on-the-fly to play next, and hit 'back' to revisit tracks they just heard.

### 📜 Formal Rules & Requirements:
1. **Up Next Priority:** Songs in the *Up Next* queue play strictly **before** the playlist continues.
2. **Playlist Position Invariant:** Playing from *Up Next* or *history* **does not move the playlist position**.
3. **Bounded History (Max 10):** History keeps the last **10 songs played** and **drops the oldest** in $O(1)$.
4. **Back Operation Semantics:** `back()` replays the top song and **removes it from history** without moving playlist position.
5. **Required Operations:**
   - `addToUpNext(songId)`
   - `playNext()`
   - `back()`


## 🏗️ Architecture & Control Flow

```
                      +---------------------------------------+
                      |         TuneWave Player Engine        |
                      +---------------------------------------+
                                          |
                +-------------------------+-------------------------+
                |                                                   |
                v                                                   v
   [ UP NEXT QUEUE (FIFO) ]                             [ PLAY HISTORY STACK (LIFO) ]
   (Takes Playback Precedence)                          (Capacity: 10, Drops Oldest)
   
   [In] -> [Song U2] [Song U1] -> [Out]                      [Top (Most Recent)]
                                                                     |
                                                                     v
                                                                +---------+
                                                                | Song P2 | (Popped by back())
                                                                +---------+
                                                                | Song P1 |
                                                                +---------+
                                                                |   ...   |
                                                                +---------+
                                                             [Bottom (Oldest)]
                                                            (Dropped if len > 10)

   -------------------------------------------------------------------------
   CRITICAL INVARIANT:
   Playing from Up Next or History leaves playlist.current completely frozen!
   -------------------------------------------------------------------------
```


In [1]:
from collections import deque
import sys
import os

# Import Milestone 1 classes
from milestone1 import Song, Playlist, SongNode


class UpNextQueue:
    """FIFO Queue for Up Next tracks with O(1) enqueue and dequeue."""
    def __init__(self):
        self._queue = deque()

    def enqueue(self, song: Song):
        self._queue.append(song)

    def dequeue(self) -> Song:
        if not self._queue:
            return None
        return self._queue.popleft()

    def peek(self) -> Song:
        return self._queue[0] if self._queue else None

    def is_empty(self) -> bool:
        return len(self._queue) == 0

    def __len__(self):
        return len(self._queue)

    def to_list(self):
        return list(self._queue)


class PlayHistory:
    """
    Bounded LIFO Stack for recently played songs.
    Rule: History keeps the last 10 songs played and drops the oldest in O(1).
    """
    def __init__(self, capacity: int = 10):
        self.capacity = capacity
        self._stack = deque(maxlen=capacity)

    def push(self, song: Song):
        """Pushes song to top. If size > capacity, oldest at bottom is dropped automatically."""
        self._stack.append(song)

    def pop(self) -> Song:
        """Pops and returns top song from history."""
        if not self._stack:
            return None
        return self._stack.pop()

    def peek(self) -> Song:
        return self._stack[-1] if self._stack else None

    def is_empty(self) -> bool:
        return len(self._stack) == 0

    def __len__(self):
        return len(self._stack)

    def to_list(self):
        return list(self._stack)


class TuneWavePlayer:
    """
    Integrated Music Player implementing Milestone 1 & 2 playback rules.
    """
    def __init__(self, playlist: Playlist = None, catalog: dict = None):
        self.playlist = playlist if playlist is not None else Playlist()
        self.up_next = UpNextQueue()
        self.history = PlayHistory(capacity=10)
        self.catalog = catalog if catalog is not None else {}
        
        # Populate catalog with playlist songs
        if playlist:
            for s_id, node in playlist.song_map.items():
                self.catalog[s_id] = node.song

        self.currently_playing = self.playlist.getCurrentSong()

    def registerSong(self, song: Song):
        self.catalog[song.song_id] = song

    def addToUpNext(self, songId) -> bool:
        """
        Queues a song to play next.
        """
        song = None
        if isinstance(songId, Song):
            song = songId
        else:
            s_id = str(songId)
            if s_id in self.catalog:
                song = self.catalog[s_id]
            elif self.playlist and s_id in self.playlist.song_map:
                song = self.playlist.song_map[s_id].song

        if song is None:
            return False

        self.up_next.enqueue(song)
        return True

    def playNext(self):
        """
        Priority rules:
        1. Songs in Up Next queue play before the playlist continues.
        2. Playing from Up Next does NOT move playlist position.
        3. History records the previously played song (max 10).
        """
        # 1. Check Up Next Queue
        if not self.up_next.is_empty():
            next_song = self.up_next.dequeue()
            if self.currently_playing is not None:
                self.history.push(self.currently_playing)
            
            self.currently_playing = next_song
            # Playlist position is NOT moved!
            return self.currently_playing

        # 2. Sequential Playlist
        if self.playlist is None or len(self.playlist) == 0:
            return "end"

        if self.currently_playing is None:
            self.currently_playing = self.playlist.getCurrentSong()
            return self.currently_playing

        res = self.playlist.next()
        if res == "end":
            return "end"

        self.history.push(self.currently_playing)
        self.currently_playing = res
        return self.currently_playing

    def back(self):
        """
        Rules:
        - back() replays the top song and removes it from history.
        - Playing from history does NOT move the playlist position.
        """
        if self.history.is_empty():
            return None

        replayed_song = self.history.pop()
        self.currently_playing = replayed_song
        # Playlist position does NOT move!
        return self.currently_playing

    def status(self) -> str:
        curr_p = self.playlist.getCurrentSong() if self.playlist else None
        p_str = f"'{curr_p.title}' [{curr_p.song_id}]" if curr_p else "None"
        curr_play = f"'{self.currently_playing.title}' [{self.currently_playing.song_id}]" if self.currently_playing else "None"
        queue_titles = [s.title for s in self.up_next.to_list()]
        history_titles = [s.title for s in self.history.to_list()]
        
        return (
            f"┌─ TuneWave Player Status ─────────────────────────────────────────\n"
            f"│  Active Track:     {curr_play}\n"
            f"│  Playlist Cursor:  {p_str}\n"
            f"│  Up Next ({len(self.up_next)}):     {queue_titles}\n"
            f"│  History ({len(self.history)}/10):  {history_titles}\n"
            f"└──────────────────────────────────────────────────────────────────"
        )


### 🧪 Interactive Demo 1: Up Next Priority & Cursor Immutability
Demonstrating that Up Next tracks play before playlist tracks, and the playlist position is preserved.


In [2]:
# Build album playlist
album = Playlist("After Hours")
p1 = Song("P1", "Alone Again", "The Weeknd")
p2 = Song("P2", "Too Late", "The Weeknd")
p3 = Song("P3", "Hardest to Love", "The Weeknd")
album.addSong(p1)
album.addSong(p2)
album.addSong(p3)

player = TuneWavePlayer(album)

# Extra singles
u1 = Song("U1", "Bad Guy", "Billie Eilish")
u2 = Song("U2", "Flowers", "Miley Cyrus")
player.registerSong(u1)
player.registerSong(u2)

print("Initial State:")
print(player.status())

# Advance playlist once to P2
print("\n-- Advancing to P2 --")
print("Playing:", player.playNext())
print(player.status())

# User queues two songs into Up Next
print("\n-- Adding U1 and U2 to Up Next --")
player.addToUpNext("U1")
player.addToUpNext("U2")
print(player.status())

# Next plays U1 from Up Next
print("\n-- Calling playNext() (should play U1, playlist cursor stays on P2) --")
print("Playing:", player.playNext())
print(player.status())

# Next plays U2 from Up Next
print("\n-- Calling playNext() (should play U2, playlist cursor stays on P2) --")
print("Playing:", player.playNext())
print(player.status())

# Next plays P3 from playlist (Up Next now empty)
print("\n-- Calling playNext() (Up Next exhausted -> resumes playlist at P3) --")
print("Playing:", player.playNext())
print(player.status())


Initial State:
┌─ TuneWave Player Status ─────────────────────────────────────────
│  Active Track:     'Alone Again' [P1]
│  Playlist Cursor:  'Alone Again' [P1]
│  Up Next (0):     []
│  History (0/10):  []
└──────────────────────────────────────────────────────────────────

-- Advancing to P2 --
Playing: 'Too Late' by The Weeknd [P2]
┌─ TuneWave Player Status ─────────────────────────────────────────
│  Active Track:     'Too Late' [P2]
│  Playlist Cursor:  'Too Late' [P2]
│  Up Next (0):     []
│  History (1/10):  ['Alone Again']
└──────────────────────────────────────────────────────────────────

-- Adding U1 and U2 to Up Next --
┌─ TuneWave Player Status ─────────────────────────────────────────
│  Active Track:     'Too Late' [P2]
│  Playlist Cursor:  'Too Late' [P2]
│  Up Next (2):     ['Bad Guy', 'Flowers']
│  History (1/10):  ['Alone Again']
└──────────────────────────────────────────────────────────────────

-- Calling playNext() (should play U1, playlist cursor stays on P2)

### ⏪ Interactive Demo 2: History Rewind (`back()`)
Testing `back()` popping songs from history and verifying playlist cursor does not move.


In [3]:
print("State before back():")
print(player.status())

# 1st back() -> replays U2
print("\n-- 1st back() call --")
print("Replaying:", player.back())
print(player.status())

# 2nd back() -> replays U1
print("\n-- 2nd back() call --")
print("Replaying:", player.back())
print(player.status())

# 3rd back() -> replays P2
print("\n-- 3rd back() call --")
print("Replaying:", player.back())
print(player.status())


State before back():
┌─ TuneWave Player Status ─────────────────────────────────────────
│  Active Track:     'Hardest to Love' [P3]
│  Playlist Cursor:  'Hardest to Love' [P3]
│  Up Next (0):     []
│  History (4/10):  ['Alone Again', 'Too Late', 'Bad Guy', 'Flowers']
└──────────────────────────────────────────────────────────────────

-- 1st back() call --
Replaying: 'Flowers' by Miley Cyrus [U2]
┌─ TuneWave Player Status ─────────────────────────────────────────
│  Active Track:     'Flowers' [U2]
│  Playlist Cursor:  'Hardest to Love' [P3]
│  Up Next (0):     []
│  History (3/10):  ['Alone Again', 'Too Late', 'Bad Guy']
└──────────────────────────────────────────────────────────────────

-- 2nd back() call --
Replaying: 'Bad Guy' by Billie Eilish [U1]
┌─ TuneWave Player Status ─────────────────────────────────────────
│  Active Track:     'Bad Guy' [U1]
│  Playlist Cursor:  'Hardest to Love' [P3]
│  Up Next (0):     []
│  History (2/10):  ['Alone Again', 'Too Late']
└──────────────

### 📦 Interactive Demo 3: Bounded History (Capacity = 10) & Oldest Eviction
Validating that history retains at most 10 songs and drops the oldest in $O(1)$ time.


In [4]:
print("Simulating playing 15 tracks to verify eviction...")
test_player = TuneWavePlayer()

for i in range(1, 16):
    track = Song(f"TRACK_{i}", f"Track Title {i}", "Artist")
    test_player.registerSong(track)
    test_player.addToUpNext(track)

# Play all 15 tracks
for i in range(15):
    test_player.playNext()

print(f"Total songs played: 15")
print(f"History stack size: {len(test_player.history)} (Must be <= 10)")
print("History contents (oldest to newest):")
for idx, s in enumerate(test_player.history.to_list(), 1):
    print(f"  {idx}. {s.title} [{s.song_id}]")

# Verify oldest (TRACK_1 through TRACK_4) were dropped
oldest_in_history = test_player.history.to_list()[0]
newest_in_history = test_player.history.peek()
print(f"\nOldest surviving song in history: {oldest_in_history.title} (TRACK_5)")
print(f"Newest song at top of history: {newest_in_history.title} (TRACK_14)")
assert len(test_player.history) == 10
assert oldest_in_history.song_id == "TRACK_5"
assert newest_in_history.song_id == "TRACK_14"
print("✓ Bounded history eviction verified!")


Simulating playing 15 tracks to verify eviction...
Total songs played: 15
History stack size: 10 (Must be <= 10)
History contents (oldest to newest):
  1. Track Title 5 [TRACK_5]
  2. Track Title 6 [TRACK_6]
  3. Track Title 7 [TRACK_7]
  4. Track Title 8 [TRACK_8]
  5. Track Title 9 [TRACK_9]
  6. Track Title 10 [TRACK_10]
  7. Track Title 11 [TRACK_11]
  8. Track Title 12 [TRACK_12]
  9. Track Title 13 [TRACK_13]
  10. Track Title 14 [TRACK_14]

Oldest surviving song in history: Track Title 5 (TRACK_5)
Newest song at top of history: Track Title 14 (TRACK_14)
✓ Bounded history eviction verified!


### 🛡️ Automated Comprehensive Test Suite for Milestone 2
Validating all invariants with programmatic assertions.


In [5]:
def run_milestone2_tests():
    p = Playlist("Test Playlist")
    t1 = Song("T1", "Track 1", "Artist")
    t2 = Song("T2", "Track 2", "Artist")
    t3 = Song("T3", "Track 3", "Artist")
    p.addSong(t1)
    p.addSong(t2)
    p.addSong(t3)

    pl = TuneWavePlayer(p)
    u1 = Song("U1", "Queue 1", "Artist")
    u2 = Song("U2", "Queue 2", "Artist")
    pl.registerSong(u1)
    pl.registerSong(u2)

    # 1. Initial State
    assert pl.currently_playing == t1
    assert pl.playlist.getCurrentSong() == t1
    assert len(pl.history) == 0

    # 2. Advance to T2
    assert pl.playNext() == t2
    assert pl.playlist.getCurrentSong() == t2
    assert pl.history.peek() == t1

    # 3. Add to Up Next
    assert pl.addToUpNext("U1") is True
    assert pl.addToUpNext("U2") is True
    assert len(pl.up_next) == 2

    # 4. Play from Up Next -> playlist position must NOT move
    assert pl.playNext() == u1
    assert pl.playlist.getCurrentSong() == t2  # FROZEN!
    assert pl.history.peek() == t2

    assert pl.playNext() == u2
    assert pl.playlist.getCurrentSong() == t2  # FROZEN!
    assert pl.history.peek() == u1

    # 5. Exhaust Up Next -> resumes playlist at T3
    assert pl.playNext() == t3
    assert pl.playlist.getCurrentSong() == t3
    assert pl.history.peek() == u2

    # 6. Back() operations
    assert pl.back() == u2
    assert pl.playlist.getCurrentSong() == t3  # FROZEN!
    assert pl.back() == u1
    assert pl.playlist.getCurrentSong() == t3  # FROZEN!
    assert pl.back() == t2
    assert pl.playlist.getCurrentSong() == t3  # FROZEN!
    assert pl.back() == t1
    assert pl.playlist.getCurrentSong() == t3  # FROZEN!
    assert pl.back() is None  # History empty now

    # 7. Bounded History maxlen=10
    h_player = TuneWavePlayer()
    for i in range(20):
        s = Song(f"S{i}", f"Song {i}", "Artist")
        h_player.history.push(s)
    assert len(h_player.history) == 10
    assert h_player.history.to_list()[0].song_id == "S10"
    assert h_player.history.peek().song_id == "S19"

    print("🎉 ALL MILESTONE 2 TESTS PASSED WITH 100% SUCCESS!")

run_milestone2_tests()


🎉 ALL MILESTONE 2 TESTS PASSED WITH 100% SUCCESS!


## 📊 Complexity & Invariant Summary

| Operation | Data Structure | Time Complexity | Space Complexity |
| :--- | :--- | :---: | :---: |
| `addToUpNext(songId)` | FIFO Queue (`deque`) | **$O(1)$** | **$O(1)$** |
| `playNext()` (from Up Next) | FIFO Dequeue + Stack Push | **$O(1)$** | **$O(1)$** |
| `playNext()` (from Playlist) | DLL Next + Stack Push | **$O(1)$** | **$O(1)$** |
| `back()` | LIFO Stack Pop | **$O(1)$** | **$O(1)$** |
| History Overflow Eviction | Double-Ended Ring Buffer | **$O(1)$** | **$O(1)$** |

Milestone 2 is complete and verified!
